# ZIP Code Food Insecurity Forecast from MMG ZCTA Panel

This notebook predicts annual ZIP/ZCTA-level food insecurity rates for the eight years after the latest observed year in `data/external/MMG_2025.xlsx`.

Unlike the earlier projection notebook, this version uses the newly added `MMG_2025.xlsx` workbook, which contains a multi-year ZCTA panel for 2020-2023. The model trains on observed year-to-year ZIP/ZCTA transitions and then recursively forecasts 2024-2031.

The default output focuses on Feeding Tampa Bay ZIP/ZCTA rows, while the model is trained on the full national ZCTA panel to improve stability. Set `TARGET_FOOD_BANK = None` to forecast every latest-year ZCTA row in the workbook.

In [1]:
# Load core libraries used throughout the notebook.
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

# Compatibility shim for the older openpyxl available in this environment.
# The installed openpyxl version still references np.float, which newer NumPy removed.
if not hasattr(np, "float"):
    np.float = float
from openpyxl import load_workbook

# Make wide dataframes easier to inspect in notebook output.
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:.4f}".format)


/opt/anaconda3/lib/python3.8/site-packages/pandas/core/computation/expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Configuration

In [2]:
# Resolve project paths so the notebook works from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "external").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Main source workbook and sheet added to data/external.
MMG_WORKBOOK_PATH = EXTERNAL_DIR / "MMG_2025.xlsx"
ZCTA_SHEET = "ZCTA"

# Default: forecast Feeding Tampa Bay rows. Set to None to forecast all latest-year ZCTA rows.
TARGET_FOOD_BANK = "Feeding Tampa Bay"

# Forecast 2024-2031 because 2023 is the latest observed year in the workbook.
FORECAST_HORIZON_YEARS = 8
OUTPUT_PATH = PROCESSED_DIR / "zipcode_food_insecurity_forecast_mmg_panel_2024_2031.csv"

MMG_WORKBOOK_PATH, OUTPUT_PATH


(PosixPath('/Users/leekho_1/ftb/data/external/MMG_2025.xlsx'),
 PosixPath('/Users/leekho_1/ftb/data/processed/zipcode_food_insecurity_forecast_mmg_panel_2024_2031.csv'))

## Helper Functions

In [3]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize spreadsheet column names for consistent downstream access."""
    out = df.copy()
    out.columns = [re.sub(r"\s+", " ", str(c).strip()) if c is not None else f"unnamed_{i}" for i, c in enumerate(out.columns)]
    return out


def read_excel_sheet_openpyxl(path: Path, sheet_name: str) -> pd.DataFrame:
    """Read a workbook sheet directly with openpyxl to avoid pandas/openpyxl version issues."""
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)

    # Drop trailing blank workbook columns so they do not become unusable dataframe fields.
    useful_cols = [i for i, value in enumerate(header) if value is not None]
    names = [header[i] for i in useful_cols]
    data = [[row[i] for i in useful_cols] for row in rows]
    return clean_columns(pd.DataFrame(data, columns=names))


def parse_number(series: pd.Series) -> pd.Series:
    """Convert currency, comma-formatted, and percent-looking strings to numeric values."""
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_rate(series: pd.Series) -> pd.Series:
    """Convert rate columns to decimals, e.g. 15.2% or 15.2 becomes 0.152."""
    values = parse_number(series)
    if values.dropna().max() > 1.5:
        values = values / 100.0
    return values


def clip_rate(series: pd.Series, low: float = 0.0, high: float = 0.95) -> pd.Series:
    """Keep modeled rates inside a plausible bounded interval."""
    return series.clip(lower=low, upper=high)


def fit_weighted_ridge(X: pd.DataFrame, y: pd.Series, weights: pd.Series, alpha: float = 1.0) -> dict:
    """Fit a population-weighted ridge regression using only NumPy."""
    # Standardize features so the ridge penalty treats each predictor comparably.
    means = X.mean()
    stds = X.std(ddof=0).replace(0, 1)
    X_scaled = (X - means) / stds

    # Add an intercept column and apply square-root weights for weighted least squares.
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    y_values = y.to_numpy(dtype=float)
    w = np.sqrt(np.maximum(weights.fillna(weights.median()).to_numpy(dtype=float), 1.0))
    Xw = X_design * w[:, None]
    yw = y_values * w

    # Penalize feature coefficients but not the intercept.
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0
    coef = np.linalg.solve(Xw.T @ Xw + penalty, Xw.T @ yw)
    return {"features": list(X.columns), "means": means, "stds": stds, "coef": coef}


def predict_weighted_ridge(model: dict, X: pd.DataFrame) -> pd.Series:
    """Generate predictions from the fitted weighted ridge model."""
    X_scaled = (X[model["features"]] - model["means"]) / model["stds"]
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    return pd.Series(X_design @ model["coef"], index=X.index)


def add_linear_feature_forecasts(history: pd.DataFrame, baseline: pd.DataFrame, feature_cols: list, forecast_years: list) -> pd.DataFrame:
    """Project each future driver with a per-row linear trend from observed history."""
    slope_rows = []

    # Estimate one slope per ZIP-county row and per driver using available observed years.
    for key, group in history.groupby("row_id"):
        years = group["year"].to_numpy(dtype=float)
        item = {"row_id": key}
        for col in feature_cols:
            y = group[col].to_numpy(dtype=float)
            valid = np.isfinite(years) & np.isfinite(y)
            if valid.sum() >= 2:
                item[f"{col}_slope"] = np.polyfit(years[valid], y[valid], deg=1)[0]
            else:
                item[f"{col}_slope"] = 0.0
        slope_rows.append(item)

    slopes = pd.DataFrame(slope_rows)
    out = baseline.merge(slopes, on="row_id", how="left")
    frames = []

    # Start from the latest observed row and add slope * years-ahead for each future year.
    for year in forecast_years:
        f = out.copy()
        f["year"] = year
        step = year - int(baseline["year"].max())
        for col in feature_cols:
            f[col] = f[col] + f[f"{col}_slope"].fillna(0) * step
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


## Load the MMG ZCTA Panel

In [4]:
# Load the national ZCTA panel from the MMG workbook.
zcta_raw = read_excel_sheet_openpyxl(MMG_WORKBOOK_PATH, ZCTA_SHEET)
print(f"Raw ZCTA rows: {len(zcta_raw):,}")

# Display a quick sample and a few year/state/food-bank combinations for sanity checking.
display(zcta_raw.head())
display(zcta_raw[["Year", "State", "Food Bank 1"]].drop_duplicates().head())


Raw ZCTA rows: 185,919


,State FIPS,County FIPS,ZCTA,Geography,"County, State",State,Year,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,Total Population (5 Year ACS),Overall Food Insecurity Rate,# of Food Insecure Persons Overall,Unemployment Rate (1 Yr BLS),Poverty Rate (5 Yr ACS),Percent Black (5 Yr ACS),Percent Hispanic (any race) (5 Year ACS),Median Income (5 Yr ACS),Homeownership Rate (5 Yr ACS),Disability Rate (5 Yr ACS)
0,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2020.0000,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,1830.0000,0.1500,270.0000,0.0480,0.1740,0.5960,0.0000,22292.0000,0.8130,0.2780
1,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2021.0000,54.0000,Montgomery Area Food Bank,NaN,None,2397.0000,0.1590,380.0000,0.0750,0.2050,0.6080,0.0030,28023.0000,0.8060,0.2490
2,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2022.0000,54.0000,Heart of Alabama Food Bank,NaN,None,2420.0000,0.1640,400.0000,0.0730,0.1900,0.6640,0.0050,31719.0000,0.8230,0.1670
3,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,2023.0000,54.0000,Heart of Alabama Food Bank,NaN,None,2555.0000,0.1380,350.0000,0.0580,0.1350,0.7190,0.0040,37938.0000,0.8690,0.1070
4,01,01001,36006,ZCTA5 36006,"Autauga County, Alabama",AL,2020.0000,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,1571.0000,0.1540,240.0000,0.0120,0.2340,0.1170,0.0110,19792.0000,0.9000,0.1760


,Year,State,Food Bank 1
0,2020.0000,AL,"Montgomery Area Food Bank, Inc."
1,2021.0000,AL,Montgomery Area Food Bank
2,2022.0000,AL,Heart of Alabama Food Bank
3,2023.0000,AL,Heart of Alabama Food Bank
43,2020.0000,AL,Feeding the Gulf Coast


## Prepare Modeling Table

The row identity is `State FIPS + County FIPS + ZCTA`, because some ZCTAs cross county boundaries and appear as more than one ZIP-county row.

In [5]:
# Work on a copy so the raw import remains untouched.
zcta = zcta_raw.copy()

# Normalize identifiers. ZCTAs can cross county boundaries, so row_id keeps county and ZIP together.
zcta["year"] = pd.to_numeric(zcta["Year"], errors="coerce").astype("Int64")
zcta["state_fips"] = zcta["State FIPS"].astype(str).str.zfill(2)
zcta["county_fips"] = zcta["County FIPS"].astype(str).str.zfill(5)
zcta["zcta"] = zcta["ZCTA"].astype(str).str.zfill(5)
zcta["row_id"] = zcta["state_fips"] + "_" + zcta["county_fips"] + "_" + zcta["zcta"]

# Parse target, count, population, and driver columns into numeric modeling fields.
zcta["population"] = parse_number(zcta["Total Population (5 Year ACS)"])
zcta["food_insecurity_rate"] = parse_rate(zcta["Overall Food Insecurity Rate"])
zcta["food_insecure_persons"] = parse_number(zcta["# of Food Insecure Persons Overall"])
zcta["unemployment_rate"] = parse_rate(zcta["Unemployment Rate (1 Yr BLS)"])
zcta["poverty_rate"] = parse_rate(zcta["Poverty Rate (5 Yr ACS)"])
zcta["percent_black"] = parse_rate(zcta["Percent Black (5 Yr ACS)"])
zcta["percent_hispanic"] = parse_rate(zcta["Percent Hispanic (any race) (5 Year ACS)"])
zcta["median_income"] = parse_number(zcta["Median Income (5 Yr ACS)"])
zcta["log_median_income"] = np.log(zcta["median_income"].replace(0, np.nan))
zcta["homeownership_rate"] = parse_rate(zcta["Homeownership Rate (5 Yr ACS)"])
zcta["disability_rate"] = parse_rate(zcta["Disability Rate (5 Yr ACS)"])

# Keep only fields needed for modeling and reporting.
model_cols = [
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year", "population",
    "food_insecurity_rate", "food_insecure_persons", "unemployment_rate", "poverty_rate",
    "percent_black", "percent_hispanic", "median_income", "log_median_income",
    "homeownership_rate", "disability_rate"
]
panel = zcta[model_cols].dropna(subset=["year", "row_id"]).copy()
panel["year"] = panel["year"].astype(int)
panel = panel.sort_values(["row_id", "year"])

# Fill driver gaps with state medians first, then national medians as a final fallback.
numeric_features = ["unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income", "homeownership_rate", "disability_rate"]
for col in numeric_features:
    panel[col] = panel.groupby("State")[col].transform(lambda s: s.fillna(s.median()))
    panel[col] = panel[col].fillna(panel[col].median())

# Derive forecast years dynamically from the latest observed year and configured horizon.
latest_year = int(panel["year"].max())
forecast_years = list(range(latest_year + 1, latest_year + 1 + FORECAST_HORIZON_YEARS))
print(f"Observed years: {sorted(panel['year'].dropna().unique().tolist())}")
print(f"Latest observed year: {latest_year}")
print(f"Forecast years: {forecast_years}")
display(panel.head())


Observed years: [2020, 2021, 2022, 2023]
Latest observed year: 2023
Forecast years: [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031]


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,year,population,food_insecurity_rate,food_insecure_persons,unemployment_rate,poverty_rate,percent_black,percent_hispanic,median_income,log_median_income,homeownership_rate,disability_rate
0,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,2020,1830.0000,0.1500,270.0000,0.0480,0.1740,0.5960,0.0000,22292.0000,10.0120,0.8130,0.2780
1,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Montgomery Area Food Bank,NaN,None,2021,2397.0000,0.1590,380.0000,0.0750,0.2050,0.6080,0.0030,28023.0000,10.2408,0.8060,0.2490
2,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Heart of Alabama Food Bank,NaN,None,2022,2420.0000,0.1640,400.0000,0.0730,0.1900,0.6640,0.0050,31719.0000,10.3647,0.8230,0.1670
3,01_01001_36003,01,01001,36003,ZCTA5 36003,"Autauga County, Alabama",AL,54.0000,Heart of Alabama Food Bank,NaN,None,2023,2555.0000,0.1380,350.0000,0.0580,0.1350,0.7190,0.0040,37938.0000,10.5437,0.8690,0.1070
4,01_01001_36006,01,01001,36006,ZCTA5 36006,"Autauga County, Alabama",AL,54.0000,"Montgomery Area Food Bank, Inc.",NaN,None,2020,1571.0000,0.1540,240.0000,0.0120,0.2340,0.1170,0.0110,19792.0000,9.8930,0.9000,0.1760


## Train a ZIP-Year Transition Model

The model predicts a ZIP-county row's food insecurity rate in year `t` using its prior-year food insecurity rate plus current economic and demographic drivers.

In [6]:
# Build the transition-model target structure: current-year rate predicted from prior-year rate.
panel["lag_food_insecurity_rate"] = panel.groupby("row_id")["food_insecurity_rate"].shift(1)
panel["year_centered"] = panel["year"] - latest_year

feature_cols = [
    "lag_food_insecurity_rate", "unemployment_rate", "poverty_rate", "percent_black",
    "percent_hispanic", "log_median_income", "homeownership_rate", "disability_rate", "year_centered"
]

# Training rows need both the current-year target and the prior-year food insecurity rate.
train = panel.dropna(subset=["food_insecurity_rate", "lag_food_insecurity_rate", "population"]).copy()

# Tune the ridge alpha with a time-based holdout.
# We train on pre-latest-year transitions and score predictions for the latest observed year.
ALPHA_GRID = [0.01, 0.1, 0.5, 1.0, 2.0]

alpha_tuning_train = train.loc[train["year"] < latest_year].copy()
alpha_tuning_eval = panel.loc[
    panel["year"].eq(latest_year)
    & panel["food_insecurity_rate"].notna()
    & panel["lag_food_insecurity_rate"].notna()
    & panel["population"].notna()
].copy()

if TARGET_FOOD_BANK:
    alpha_tuning_local_eval = alpha_tuning_eval.loc[
        alpha_tuning_eval["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)
    ].copy()
else:
    alpha_tuning_local_eval = alpha_tuning_eval.copy()


def weighted_alpha_metrics(frame: pd.DataFrame, prediction_col: str = "alpha_prediction") -> dict:
    """Calculate weighted validation metrics for alpha tuning."""
    if frame.empty:
        return {"weighted_mae": np.nan, "weighted_rmse": np.nan, "weighted_mean_error": np.nan}
    weights = frame["population"].fillna(1).clip(lower=1)
    errors = frame[prediction_col] - frame["food_insecurity_rate"]
    return {
        "weighted_mae": np.average(errors.abs(), weights=weights),
        "weighted_rmse": np.sqrt(np.average(errors ** 2, weights=weights)),
        "weighted_mean_error": np.average(errors, weights=weights),
    }

alpha_results = []
for alpha in ALPHA_GRID:
    candidate_model = fit_weighted_ridge(
        X=alpha_tuning_train[feature_cols],
        y=alpha_tuning_train["food_insecurity_rate"],
        weights=alpha_tuning_train["population"],
        alpha=alpha,
    )
    scored = alpha_tuning_eval.copy()
    scored["alpha_prediction"] = clip_rate(predict_weighted_ridge(candidate_model, scored[feature_cols]))
    local_scored = scored.loc[scored.index.isin(alpha_tuning_local_eval.index)].copy()

    national_metrics = weighted_alpha_metrics(scored)
    local_metrics = weighted_alpha_metrics(local_scored)
    alpha_results.append({
        "alpha": alpha,
        "national_weighted_mae": national_metrics["weighted_mae"],
        "national_weighted_rmse": national_metrics["weighted_rmse"],
        "national_weighted_mean_error": national_metrics["weighted_mean_error"],
        "local_weighted_mae": local_metrics["weighted_mae"],
        "local_weighted_rmse": local_metrics["weighted_rmse"],
        "local_weighted_mean_error": local_metrics["weighted_mean_error"],
    })

alpha_tuning_results = pd.DataFrame(alpha_results)
selection_metric = "local_weighted_mae" if alpha_tuning_results["local_weighted_mae"].notna().any() else "national_weighted_mae"
selected_alpha = float(
    alpha_tuning_results
    .sort_values([selection_metric, "national_weighted_mae", "alpha"], ascending=[True, True, True])
    .iloc[0]["alpha"]
)

print(f"Selected alpha: {selected_alpha:g} using {selection_metric}")
display(alpha_tuning_results)

# Fit the population-weighted ridge transition model.
model = fit_weighted_ridge(
    X=train[feature_cols],
    y=train["food_insecurity_rate"],
    weights=train["population"],
    alpha=selected_alpha,
)

# In-sample diagnostics provide a rough reasonableness check, not a formal validation study.
train["prediction"] = clip_rate(predict_weighted_ridge(model, train[feature_cols]))
weighted_mae = np.average(np.abs(train["food_insecurity_rate"] - train["prediction"]), weights=train["population"])
weighted_rmse = np.sqrt(np.average((train["food_insecurity_rate"] - train["prediction"]) ** 2, weights=train["population"]))
residual_std = float(np.std(train["food_insecurity_rate"] - train["prediction"], ddof=1))

print(f"Training rows: {len(train):,}")
print(f"Weighted MAE: {weighted_mae:.4f}")
print(f"Weighted RMSE: {weighted_rmse:.4f}")
print(f"Residual std: {residual_std:.4f}")
display(pd.DataFrame({"feature": ["intercept"] + feature_cols, "coefficient": model["coef"]}))

# Separate same-year model used only to fill missing latest-year baselines.
# This model does not drive the recursive forecast except where 2023 baseline data is absent.
baseline_feature_cols = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic",
    "log_median_income", "homeownership_rate", "disability_rate", "year_centered"
]
baseline_train = panel.dropna(subset=["food_insecurity_rate", "population"]).copy()
baseline_model = fit_weighted_ridge(
    X=baseline_train[baseline_feature_cols],
    y=baseline_train["food_insecurity_rate"],
    weights=baseline_train["population"],
    alpha=selected_alpha,
)
panel["baseline_model_prediction"] = clip_rate(predict_weighted_ridge(baseline_model, panel[baseline_feature_cols]))


Selected alpha: 2 using local_weighted_mae


,alpha,national_weighted_mae,national_weighted_rmse,national_weighted_mean_error,local_weighted_mae,local_weighted_rmse,local_weighted_mean_error
0,0.0100,0.0395,0.0408,0.0395,0.0409,0.0415,0.0409
1,0.1000,0.0395,0.0408,0.0395,0.0409,0.0415,0.0409
2,0.5000,0.0395,0.0408,0.0395,0.0409,0.0415,0.0409
3,1.0000,0.0395,0.0408,0.0395,0.0409,0.0415,0.0409
4,2.0000,0.0395,0.0408,0.0395,0.0409,0.0415,0.0409


Training rows: 109,633
Weighted MAE: 0.0105
Weighted RMSE: 0.0129
Residual std: 0.0153


,feature,coefficient
0,intercept,0.1336
1,lag_food_insecurity_rate,0.0229
2,unemployment_rate,0.0079
3,poverty_rate,0.0163
4,percent_black,-0.0039
5,percent_hispanic,-0.0002
6,log_median_income,-0.0024
7,homeownership_rate,-0.0038
8,disability_rate,0.0070
9,year_centered,0.0121


## Local Validation on Feeding Tampa Bay Rows

This section evaluates how well the model predicts the latest observed year for Feeding Tampa Bay rows. It fits a temporary validation model on earlier transitions only, then predicts the latest observed year using the known prior-year food insecurity rate and current-year drivers.

These diagnostics do not change the production forecast model; they are a local reasonableness check for the target service area.

In [7]:
# Hold out the latest observed year to evaluate local performance on Feeding Tampa Bay rows.
validation_train = train.loc[train["year"] < latest_year].copy()
validation_eval = panel.loc[
    panel["year"].eq(latest_year)
    & panel["food_insecurity_rate"].notna()
    & panel["lag_food_insecurity_rate"].notna()
    & panel["population"].notna()
].copy()

# Fit the same transition model structure, but only on pre-latest-year transitions.
validation_model = fit_weighted_ridge(
    X=validation_train[feature_cols],
    y=validation_train["food_insecurity_rate"],
    weights=validation_train["population"],
    alpha=selected_alpha,
)
validation_eval["validation_prediction"] = clip_rate(predict_weighted_ridge(validation_model, validation_eval[feature_cols]))
validation_eval["validation_error"] = validation_eval["validation_prediction"] - validation_eval["food_insecurity_rate"]
validation_eval["absolute_error"] = validation_eval["validation_error"].abs()

# Local validation subset: same rows the forecast targets by default.
if TARGET_FOOD_BANK:
    local_validation_eval = validation_eval.loc[
        validation_eval["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)
    ].copy()
else:
    local_validation_eval = validation_eval.copy()


def summarize_validation(frame: pd.DataFrame, label: str) -> dict:
    """Return weighted validation metrics for a dataframe of observed predictions."""
    if frame.empty:
        return {
            "scope": label,
            "rows": 0,
            "zcta_count": 0,
            "weighted_mae": np.nan,
            "weighted_rmse": np.nan,
            "weighted_mean_error": np.nan,
            "weighted_actual_rate": np.nan,
            "weighted_predicted_rate": np.nan,
        }
    weights = frame["population"].fillna(1).clip(lower=1)
    return {
        "scope": label,
        "rows": len(frame),
        "zcta_count": frame["zcta"].nunique(),
        "weighted_mae": np.average(frame["absolute_error"], weights=weights),
        "weighted_rmse": np.sqrt(np.average(frame["validation_error"] ** 2, weights=weights)),
        "weighted_mean_error": np.average(frame["validation_error"], weights=weights),
        "weighted_actual_rate": np.average(frame["food_insecurity_rate"], weights=weights),
        "weighted_predicted_rate": np.average(frame["validation_prediction"], weights=weights),
    }

validation_summary = pd.DataFrame([
    summarize_validation(validation_eval, f"National latest-year holdout ({latest_year})"),
    summarize_validation(local_validation_eval, f"{TARGET_FOOD_BANK or 'All'} latest-year holdout ({latest_year})"),
])

display(validation_summary)

# Show the largest local misses so they can be reviewed for data quality or local calibration needs.
local_validation_review_cols = [
    "zcta", "County, State", "Food Bank 1", "population", "lag_food_insecurity_rate",
    "food_insecurity_rate", "validation_prediction", "validation_error", "absolute_error",
    "unemployment_rate", "poverty_rate", "median_income", "homeownership_rate", "disability_rate",
]
display(
    local_validation_eval[local_validation_review_cols]
    .sort_values("absolute_error", ascending=False)
    .head(15)
)


,scope,rows,zcta_count,weighted_mae,weighted_rmse,weighted_mean_error,weighted_actual_rate,weighted_predicted_rate
0,National latest-year holdout (2023),36573,25181,0.0395,0.0408,0.0395,0.1464,0.1859
1,Feeding Tampa Bay latest-year holdout (2023),234,216,0.0409,0.0415,0.0409,0.1492,0.1901


,zcta,"County, State",Food Bank 1,population,lag_food_insecurity_rate,food_insecurity_rate,validation_prediction,validation_error,absolute_error,unemployment_rate,poverty_rate,median_income,homeownership_rate,disability_rate
26759,33849,"Polk County, Florida",Feeding Tampa Bay,736.0000,0.2550,0.1870,0.2791,0.0921,0.0921,0.1220,0.1050,69911.0000,0.9380,0.2490
26413,33849,"Pasco County, Florida",Feeding Tampa Bay,736.0000,0.2550,0.1870,0.2791,0.0921,0.0921,0.1220,0.1050,69911.0000,0.9380,0.2490
24346,33857,"Highlands County, Florida",Feeding Tampa Bay,935.0000,0.1930,0.1420,0.2276,0.0856,0.0856,0.0000,0.2140,47083.0000,0.9290,0.1130
26767,33851,"Polk County, Florida",Feeding Tampa Bay,1109.0000,0.1950,0.1710,0.2388,0.0678,0.0678,0.0000,0.2740,77557.0000,0.9090,0.1560
24262,33890,"Hardee County, Florida",Feeding Tampa Bay,4991.0000,0.2010,0.1770,0.2419,0.0649,0.0649,0.0620,0.1980,70179.0000,0.8180,0.1480
24397,33540,"Hillsborough County, Florida",Feeding Tampa Bay,11098.0000,0.1940,0.1750,0.2366,0.0616,0.0616,0.0630,0.1530,55426.0000,0.8780,0.2340
26354,33540,"Pasco County, Florida",Feeding Tampa Bay,11098.0000,0.1940,0.1750,0.2366,0.0616,0.0616,0.0630,0.1530,55426.0000,0.8780,0.2340
26417,34610,"Pasco County, Florida",Feeding Tampa Bay,16544.0000,0.1390,0.1240,0.1821,0.0581,0.0581,0.0230,0.1000,73878.0000,0.8360,0.1470
26736,33839,"Polk County, Florida",Feeding Tampa Bay,4397.0000,0.1230,0.1060,0.1632,0.0572,0.0572,0.0240,0.0750,93229.0000,0.7750,0.0880
25196,34216,"Manatee County, Florida",Feeding Tampa Bay,900.0000,0.1110,0.1000,0.1570,0.0570,0.0570,0.0000,0.0680,99188.0000,0.9530,0.1720


In [8]:
local_validation_eval[local_validation_review_cols]['validation_error'].describe()

count   234.0000
mean      0.0426
std       0.0094
min       0.0146
25%       0.0379
50%       0.0418
75%       0.0468
max       0.0921
Name: validation_error, dtype: float64

## Build Forecast Base Rows

By default, the forecast covers latest-year rows where `Food Bank 1` contains `Feeding Tampa Bay`. Change `TARGET_FOOD_BANK` above to forecast a different food bank or set it to `None` for all rows.

In [9]:
# Start forecasting from the latest observed year.
latest_rows = panel.loc[panel["year"].eq(latest_year)].copy()

# By default, keep only Feeding Tampa Bay rows; set TARGET_FOOD_BANK to None to keep all rows.
if TARGET_FOOD_BANK:
    forecast_base = latest_rows.loc[latest_rows["Food Bank 1"].astype(str).str.contains(TARGET_FOOD_BANK, case=False, na=False)].copy()
else:
    forecast_base = latest_rows.copy()

# Preserve the original 2023 value and fill missing 2023 baselines with the same-year fallback model.
forecast_base["food_insecurity_rate_2023_observed"] = forecast_base["food_insecurity_rate"]
forecast_base["food_insecurity_rate_2023_used"] = forecast_base["food_insecurity_rate"].fillna(
    forecast_base["baseline_model_prediction"]
)
forecast_base["food_insecurity_baseline_source"] = np.where(
    forecast_base["food_insecurity_rate_2023_observed"].notna(),
    "mmg_zcta_2023_observed",
    "modeled_2023_fallback",
)

print(f"Forecast base rows: {len(forecast_base):,}")
print(f"Unique ZCTAs: {forecast_base['zcta'].nunique():,}")
display(forecast_base[["zcta", "County, State", "Food Bank 1", "year", "food_insecurity_rate_2023_observed", "food_insecurity_rate_2023_used", "food_insecurity_baseline_source", "population"]].head(20))


Forecast base rows: 259
Unique ZCTAs: 241


,zcta,"County, State",Food Bank 1,year,food_insecurity_rate_2023_observed,food_insecurity_rate_2023_used,food_insecurity_baseline_source,population
23591,34428,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1970,0.1970,mmg_zcta_2023_observed,9325.0000
23595,34429,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1950,0.1950,mmg_zcta_2023_observed,9494.0000
23599,34433,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1420,0.1420,mmg_zcta_2023_observed,8218.0000
23603,34434,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1410,0.1410,mmg_zcta_2023_observed,10701.0000
23607,34436,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2160,0.2160,mmg_zcta_2023_observed,8170.0000
23611,34442,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2000,0.2000,mmg_zcta_2023_observed,16356.0000
23615,34445,"Citrus County, Florida",Feeding Tampa Bay,2023,NaN,0.0872,modeled_2023_fallback,29.0000
23619,34446,"Citrus County, Florida",Feeding Tampa Bay,2023,0.1760,0.1760,mmg_zcta_2023_observed,18797.0000
23623,34448,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2310,0.2310,mmg_zcta_2023_observed,10741.0000
23627,34449,"Citrus County, Florida",Feeding Tampa Bay,2023,0.2020,0.2020,mmg_zcta_2023_observed,3508.0000


## Project Future Driver Values

Driver values are projected with a simple per-row linear trend over the observed years. This uses the new workbook's 2020-2023 ZCTA panel. Rates are clipped to valid ranges, and income is kept positive by forecasting log income.

In [10]:
# These drivers are extrapolated into the forecast horizon and then fed into the transition model.
driver_cols = ["unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income", "homeownership_rate", "disability_rate", "population"]

# Estimate future driver values from each ZIP-county row's observed 2020-2023 trend.
future_drivers = add_linear_feature_forecasts(
    history=panel.loc[panel["row_id"].isin(forecast_base["row_id"])],
    baseline=forecast_base,
    feature_cols=driver_cols,
    forecast_years=forecast_years,
)

# Keep rate-like features in a valid range and prevent negative projected populations.
for col in ["unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "homeownership_rate", "disability_rate"]:
    future_drivers[col] = clip_rate(future_drivers[col], 0, 0.95)
future_drivers["population"] = future_drivers["population"].clip(lower=0)

# Convert log-income forecasts back into dollar-scale median income for output.
future_drivers["median_income"] = np.exp(future_drivers["log_median_income"])
display(future_drivers[["zcta", "County, State", "year", "population", "unemployment_rate", "poverty_rate", "median_income"]].head(15))


,zcta,"County, State",year,population,unemployment_rate,poverty_rate,median_income
0,34428,"Citrus County, Florida",2024,9323.4000,0.0444,0.2255,50501.6070
1,34429,"Citrus County, Florida",2024,9865.0000,0.1143,0.1569,62052.3056
2,34433,"Citrus County, Florida",2024,8636.4000,0.0207,0.1445,75700.1627
3,34434,"Citrus County, Florida",2024,11497.1000,0.0336,0.1362,58696.9406
4,34436,"Citrus County, Florida",2024,8179.5000,0.0569,0.2636,52255.0504
5,34442,"Citrus County, Florida",2024,16633.3000,0.1248,0.1487,58978.4884
6,34445,"Citrus County, Florida",2024,37.7000,0.0000,0.0000,62060.2008
7,34446,"Citrus County, Florida",2024,19136.2000,0.0606,0.1352,66266.9074
8,34448,"Citrus County, Florida",2024,10964.0000,0.0777,0.2945,47928.8259
9,34449,"Citrus County, Florida",2024,3489.1000,0.0314,0.2118,63071.2097


## Recursive Eight-Year Forecast


In [11]:
# Store the latest known or predicted food insecurity rate for each row.
# This gets updated after each forecast year so the next year can use it as the lag.
last_rate = forecast_base.set_index("row_id")["food_insecurity_rate_2023_used"].to_dict()
forecast_frames = []

for year in forecast_years:
    year_rows = future_drivers.loc[future_drivers["year"].eq(year)].copy()

    # Use the previous observed/predicted value as the lagged food insecurity rate.
    year_rows["lag_food_insecurity_rate"] = year_rows["row_id"].map(last_rate)
    year_rows["year_centered"] = year_rows["year"] - latest_year

    # Predict rate, percent, count, and simple residual-based uncertainty bands.
    year_rows["predicted_food_insecurity_rate"] = clip_rate(predict_weighted_ridge(model, year_rows[feature_cols]))
    year_rows["predicted_food_insecurity_percent"] = year_rows["predicted_food_insecurity_rate"] * 100
    year_rows["predicted_food_insecure_persons"] = (year_rows["predicted_food_insecurity_rate"] * year_rows["population"]).round()
    year_rows["uncertainty_band_low"] = clip_rate(year_rows["predicted_food_insecurity_rate"] - 1.64 * residual_std)
    year_rows["uncertainty_band_high"] = clip_rate(year_rows["predicted_food_insecurity_rate"] + 1.64 * residual_std)
    forecast_frames.append(year_rows)

    # Update the lag map so the next forecast year is recursive.
    last_rate.update(year_rows.set_index("row_id")["predicted_food_insecurity_rate"].to_dict())

forecast = pd.concat(forecast_frames, ignore_index=True)

# Select the final reporting columns and sort for easy ZIP/year review.
forecast_output = forecast[[
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year",
    "food_insecurity_rate_2023_observed", "food_insecurity_rate_2023_used", "food_insecurity_baseline_source",
    "lag_food_insecurity_rate", "predicted_food_insecurity_rate", "predicted_food_insecurity_percent",
    "predicted_food_insecure_persons", "uncertainty_band_low", "uncertainty_band_high",
    "population", "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic",
    "median_income", "homeownership_rate", "disability_rate"
]].sort_values(["zcta", "County, State", "year"])

display(forecast_output.head(20))
forecast_output.shape


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,Food Bank 2,year,food_insecurity_rate_2023_observed,food_insecurity_rate_2023_used,food_insecurity_baseline_source,lag_food_insecurity_rate,predicted_food_insecurity_rate,predicted_food_insecurity_percent,predicted_food_insecure_persons,uncertainty_band_low,uncertainty_band_high,population,unemployment_rate,poverty_rate,percent_black,percent_hispanic,median_income,homeownership_rate,disability_rate
247,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2024,0.1510,0.1510,mmg_zcta_2023_observed,0.1510,0.1771,17.7110,5493.0000,0.1521,0.2021,31011.9000,0.0476,0.0904,0.0636,0.0593,62182.3491,0.8108,0.2178
506,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2025,0.1510,0.1510,mmg_zcta_2023_observed,0.1771,0.2014,20.1401,6279.0000,0.1764,0.2264,31177.8000,0.0492,0.0888,0.0702,0.0616,67203.9166,0.8106,0.2086
765,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2026,0.1510,0.1510,mmg_zcta_2023_observed,0.2014,0.2249,22.4907,7049.0000,0.1999,0.2499,31343.7000,0.0508,0.0872,0.0768,0.0639,72631.0033,0.8104,0.1994
1024,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2027,0.1510,0.1510,mmg_zcta_2023_observed,0.2249,0.2481,24.8074,7817.0000,0.2231,0.2731,31509.6000,0.0524,0.0856,0.0834,0.0662,78496.3571,0.8102,0.1902
1283,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2028,0.1510,0.1510,mmg_zcta_2023_observed,0.2481,0.2711,27.1095,8587.0000,0.2461,0.2961,31675.5000,0.0540,0.0840,0.0900,0.0685,84835.3706,0.8100,0.1810
1542,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2029,0.1510,0.1510,mmg_zcta_2023_observed,0.2711,0.2941,29.4053,9363.0000,0.2690,0.3191,31841.4000,0.0556,0.0824,0.0966,0.0708,91686.2944,0.8098,0.1718
1801,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2030,0.1510,0.1510,mmg_zcta_2023_observed,0.2941,0.3170,31.6983,10146.0000,0.2920,0.3420,32007.3000,0.0572,0.0808,0.1032,0.0731,99090.4681,0.8096,0.1626
2060,12_12119_32159,12,12119,32159,ZCTA5 32159,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2031,0.1510,0.1510,mmg_zcta_2023_observed,0.3170,0.3399,33.9902,10936.0000,0.3149,0.3649,32173.2000,0.0588,0.0792,0.1098,0.0754,107092.5696,0.8094,0.1534
248,12_12119_32162,12,12119,32162,ZCTA5 32162,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2024,0.1310,0.1310,mmg_zcta_2023_observed,0.1310,0.1633,16.3296,8755.0000,0.1383,0.1883,53614.2000,0.0794,0.0447,0.0146,0.0212,77605.6482,0.9259,0.2236
507,12_12119_32162,12,12119,32162,ZCTA5 32162,"Sumter County, Florida",FL,90.0000,Feeding Tampa Bay,NaN,None,2025,0.1310,0.1310,mmg_zcta_2023_observed,0.1633,0.1968,19.6847,10474.0000,0.1718,0.2219,53209.4000,0.0988,0.0454,0.0172,0.0244,81467.3480,0.9178,0.2302


(2072, 29)

## Quality Checks and Save

In [12]:
# Confirm every forecast base row receives one prediction per forecast year.
expected_rows = len(forecast_base) * len(forecast_years)
actual_rows = len(forecast_output)
print(f"Expected rows: {expected_rows:,}")
print(f"Actual rows: {actual_rows:,}")
print(f"Missing predicted rates: {forecast_output['predicted_food_insecurity_rate'].isna().sum():,}")

assert actual_rows == expected_rows, "Unexpected forecast row count."
assert forecast_output["predicted_food_insecurity_rate"].between(0, 0.95).all(), "Predicted rates outside expected range."

# Summarize the forecast by year before saving.
summary_by_year = (
    forecast_output.groupby("year")
    .agg(
        row_count=("row_id", "nunique"),
        zcta_count=("zcta", "nunique"),
        mean_rate=("predicted_food_insecurity_rate", "mean"),
        population_weighted_rate=("predicted_food_insecurity_rate", lambda s: np.average(s, weights=forecast_output.loc[s.index, "population"])),
        total_predicted_food_insecure_persons=("predicted_food_insecure_persons", "sum"),
    )
)
display(summary_by_year)

# Persist the final ZIP-county-year forecast table for downstream analysis.
forecast_output.to_csv(OUTPUT_PATH, index=False)
print(f"Saved forecast to: {OUTPUT_PATH}")


Expected rows: 2,072
Actual rows: 2,072
Missing predicted rates: 0


,row_count,zcta_count,mean_rate,population_weighted_rate,total_predicted_food_insecure_persons
year,,,,,
2024,259,241,0.1773,0.1731,941542.0000
2025,259,241,0.2017,0.1964,1081635.0000
2026,259,241,0.2262,0.2196,1224797.0000
2027,259,241,0.2509,0.2429,1373367.0000
2028,259,241,0.2758,0.2663,1526615.0000
2029,259,241,0.3008,0.2900,1684629.0000
2030,259,241,0.3260,0.3138,1847549.0000
2031,259,241,0.3514,0.3379,2015562.0000


Saved forecast to: /Users/leekho_1/ftb/data/processed/zipcode_food_insecurity_forecast_mmg_panel_2024_2031.csv


## Notes and Limitations

- This model uses observed MMG ZCTA data from 2020-2023 and forecasts 2024-2031.
- Future unemployment, poverty, demographic, income, homeownership, disability, and population values are simple linear extrapolations from the observed ZCTA panel.
- The model is predictive and should not be interpreted causally.
- Uncertainty bands are residual-based diagnostics, not formal prediction intervals.
- For planning scenarios, replace the linear driver projections with externally supplied economic or demographic forecasts.